In [1]:
import pandas as pd

In [17]:
train = pd.read_csv('data/train_reviews.csv')
train.columns

Index(['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny',
       'cool', 'date'],
      dtype='str')

In [2]:
negocios = pd.read_csv('data/negocios.csv')

In [16]:
usuarios = pd.read_csv('data/usuarios.csv')
usuarios.columns

/tmp/ipykernel_326565/3874900789.py:1: DtypeWarning: Columns (0: elite) have mixed types. Specify dtype option on import or set low_memory=False.
  usuarios = pd.read_csv('data/usuarios.csv')


Index(['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny',
       'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos'],
      dtype='str')

In [3]:
negocios.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='str')

In [14]:
negocios.hours[1]

"{'Monday': '10:0-21:0', 'Tuesday': '10:0-21:0', 'Wednesday': '10:0-21:0', 'Thursday': '10:0-21:0', 'Friday': '10:0-21:0', 'Saturday': '10:0-21:0', 'Sunday': '12:0-18:0'}"

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import ast
import gc
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*100)
print("🔥 VERSIÓN 4: ONE-HOT ENCODING COMPLETO OPTIMIZADO")
print("="*100)

# ============================================
# PASO 1: USAR DATOS YA EN MEMORIA
# ============================================
print("\n📊 PASO 1: Validando datos en memoria...")

try:
    # Los datos ya están cargados en variables globales
    print(f"  ✓ usuarios_raw: {usuarios_raw.shape}")
    print(f"  ✓ negocios_raw: {negocios_raw.shape}")
    print(f"  ✓ train_reviews: {train_reviews.shape}")
    print(f"  ✓ test_reviews: {test_reviews.shape}")
except NameError as e:
    print(f"  ⚠️ Reloading data: {e}")
    usuarios_raw = pd.read_csv('data/usuarios.csv', low_memory=False)
    negocios_raw = pd.read_csv('data/negocios.csv')
    train_reviews = pd.read_csv('data/train_reviews.csv')
    test_reviews = pd.read_csv('data/test_final.csv')

# ============================================
# PASO 2: PROCESAR USUARIOS EFICIENTEMENTE
# ============================================
print("\n📊 PASO 2: Procesando usuarios...")

# Crear copia solo de columnas necesarias
usuarios_work = usuarios_raw.copy()

def extract_elite_features(elite_str):
    if pd.isna(elite_str) or elite_str == '' or str(elite_str) == 'nan':
        return 0, np.nan
    try:
        years = [int(y) for y in str(elite_str).split(',') if y.strip().isdigit()]
        return len(years), min(years) if years else np.nan
    except:
        return 0, np.nan

# Aplicar extracción en paralelo
elite_features = usuarios_work['elite'].apply(lambda x: pd.Series(extract_elite_features(x)))
usuarios_work['elite_count'] = elite_features[0]
usuarios_work['elite_from_year'] = elite_features[1]

# Friend count
usuarios_work['friend_count'] = usuarios_work['friends'].fillna('').apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) and str(x) != 'nan' and str(x) != '' else 0
)

# Days yelping
usuarios_work['yelping_since'] = pd.to_datetime(usuarios_work['yelping_since'], errors='coerce')
usuarios_work['days_yelping'] = (pd.Timestamp.now() - usuarios_work['yelping_since']).dt.days

# Total compliments
compliment_cols = [c for c in usuarios_work.columns if c.startswith('compliment_')]
usuarios_work['total_compliments'] = usuarios_work[compliment_cols].fillna(0).sum(axis=1)

# Seleccionar columnas procesadas
usuarios_processed = usuarios_work[['user_id', 'review_count', 'useful', 'funny', 'cool',
                                    'fans', 'average_stars', 'elite_count', 'friend_count',
                                    'days_yelping', 'total_compliments']].copy()

usuarios_processed['elite_count'] = usuarios_processed['elite_count'].fillna(0).astype(int)
usuarios_processed['friend_count'] = usuarios_processed['friend_count'].fillna(0).astype(int)
usuarios_processed['days_yelping'] = usuarios_processed['days_yelping'].fillna(0).astype(int)

print(f"  ✓ usuarios_processed: {usuarios_processed.shape}")
del usuarios_work
gc.collect()

# ============================================
# PASO 3: PROCESAR NEGOCIOS EFICIENTEMENTE
# ============================================
print("\n📊 PASO 3: Procesando negocios...")

negocios_work = negocios_raw.copy()

def parse_categories(cat_str):
    if pd.isna(cat_str) or cat_str == '' or str(cat_str) == 'nan':
        return []
    try:
        return [c.strip() for c in str(cat_str).split(',')]
    except:
        return []

# Identificar top 20 categorías
all_categories = negocios_work['categories'].apply(parse_categories).explode().dropna()
category_counts = Counter(all_categories)
top_20_categories = [cat for cat, _ in category_counts.most_common(20)]

print(f"  ✓ Top 20 categorías: {top_20_categories[:5]}... ({len(top_20_categories)} total)")

# Crear columnas binarias para cada categoría
for cat in top_20_categories:
    col_name = 'cat_' + cat.replace(' ', '_').replace('&', 'and').replace('/', '_').replace('-', '_')
    negocios_work[col_name] = negocios_work['categories'].apply(
        lambda x: 1 if cat in parse_categories(x) else 0
    )

# Seleccionar columnas
negocios_cols = ['business_id', 'stars', 'review_count', 'is_open', 'latitude', 'longitude', 'postal_code']
negocios_cols += [f'cat_{cat.replace(" ", "_").replace("&", "and").replace("/", "_").replace("-", "_")}' for cat in top_20_categories]

negocios_processed = negocios_work[negocios_cols].copy()
negocios_processed = negocios_processed.fillna(0)

print(f"  ✓ negocios_processed: {negocios_processed.shape}")
del negocios_work
gc.collect()

# ============================================
# PASO 4: ENSAMBLAR DATOS TRAIN
# ============================================
print("\n📊 PASO 4: Ensamblando datos TRAIN...")

X_train = train_reviews[['review_id', 'user_id', 'business_id', 'date']].copy()
X_train = X_train.merge(usuarios_processed, on='user_id', how='left')
X_train = X_train.merge(negocios_processed, on='business_id', how='left')

# Procesar fecha
X_train['date'] = pd.to_datetime(X_train['date'], errors='coerce')
X_train['year'] = X_train['date'].dt.year.fillna(0).astype(int)
X_train['month'] = X_train['date'].dt.month.fillna(0).astype(int)
X_train['day'] = X_train['date'].dt.day.fillna(0).astype(int)
X_train = X_train.drop('date', axis=1)

y_train = train_reviews['stars'].copy()

print(f"  ✓ X_train: {X_train.shape}")
print(f"  ✓ y_train: {y_train.shape}")

# ============================================
# PASO 5: ENSAMBLAR DATOS TEST
# ============================================
print("\n📊 PASO 5: Ensamblando datos TEST...")

X_test = test_reviews[['review_id', 'user_id', 'business_id', 'date']].copy() if 'date' in test_reviews.columns else test_reviews[['review_id', 'user_id', 'business_id']].copy()
X_test = X_test.merge(usuarios_processed, on='user_id', how='left')
X_test = X_test.merge(negocios_processed, on='business_id', how='left')

# Procesar fecha
if 'date' in X_test.columns:
    X_test['date'] = pd.to_datetime(X_test['date'], errors='coerce')
    X_test['year'] = X_test['date'].dt.year.fillna(0).astype(int)
    X_test['month'] = X_test['date'].dt.month.fillna(0).astype(int)
    X_test['day'] = X_test['date'].dt.day.fillna(0).astype(int)
    X_test = X_test.drop('date', axis=1)

print(f"  ✓ X_test: {X_test.shape}")

# ============================================
# PASO 6: ONE-HOT ENCODING
# ============================================
print("\n📊 PASO 6: Aplicando One-Hot Encoding...")

# Identificar columnas categóricas
exclude_cols = ['review_id', 'user_id', 'business_id', 'year', 'month', 'day']
exclude_cols += [col for col in X_train.columns if col.startswith('cat_')]  # Categorías ya binarizadas
categorical_cols = [col for col in X_train.columns if col not in exclude_cols and X_train[col].dtype == 'object']

print(f"  Columnas categóricas a codificar: {categorical_cols}")

# Aplicar OHE
if categorical_cols:
    X_train_ohe = pd.get_dummies(X_train, columns=categorical_cols, drop_first=False)
    X_test_ohe = pd.get_dummies(X_test, columns=categorical_cols, drop_first=False)
else:
    X_train_ohe = X_train.copy()
    X_test_ohe = X_test.copy()

# Alinear columnas entre train y test
train_cols = set(X_train_ohe.columns)
test_cols = set(X_test_ohe.columns)

# Agregar columnas faltantes en test
for col in train_cols - test_cols:
    X_test_ohe[col] = 0

# Eliminar columnas extra en test
for col in test_cols - train_cols:
    X_test_ohe = X_test_ohe.drop(col, axis=1, errors='ignore')

# Reordenar y asegurar mismas columnas
X_test_ohe = X_test_ohe[X_train_ohe.columns]

print(f"  ✓ X_train_ohe: {X_train_ohe.shape}")
print(f"  ✓ X_test_ohe: {X_test_ohe.shape}")

# ============================================
# PASO 7: GUARDAR ARCHIVOS
# ============================================
print("\n📊 PASO 7: Guardando archivos finales...")

# Crear train con target
train_final_ohe = pd.concat([X_train_ohe, y_train.rename('target')], axis=1)

# Garantizar que no hay valores NaN
train_final_ohe = train_final_ohe.fillna(0)
X_test_ohe = X_test_ohe.fillna(0)

# Guardar archivos
train_final_ohe.to_csv('data/train_final_ohe.csv', index=False)
X_test_ohe.to_csv('data/test_final_ohe.csv', index=False)

print(f"  ✓ train_final_ohe.csv: {train_final_ohe.shape}")
print(f"  ✓ test_final_ohe.csv: {X_test_ohe.shape}")

print("\n" + "="*100)
print("✅ PROCESAMIENTO COMPLETADO EXITOSAMENTE")
print("="*100)
print(f"\nDimensiones finales:")
print(f"  Train: {train_final_ohe.shape[0]} filas × {train_final_ohe.shape[1]} columnas")
print(f"  Test:  {X_test_ohe.shape[0]} filas × {X_test_ohe.shape[1]} columnas")
print(f"\nPrimeras columnas: {list(train_final_ohe.columns[:10])}")
print(f"Última columna: {train_final_ohe.columns[-1]}")

# Limpiar memoria
del usuarios_processed, negocios_processed, X_train, X_test
gc.collect()

print("✅ Memoria liberada. Archivos listos para usar.")
